# Phase G â€” One-click Colab / Local setup for Adaptive Autoencoder + QLoRA (Gemma3)

This notebook installs CUDA/PyTorch/TensorRT-friendly packages (Colab) or provides guidance for local WSL/Ubuntu with an RTX 3060 Ti. It then clones your repository, trains the SOM autoencoder (optionally the QLoRA adapter), runs the cluster snapshot, and demonstrates the RL feedback loop.

**Notes**:
- On Colab, use a GPU runtime (Runtime > Change runtime type > GPU).
- The notebook uses `redis-server` in-process on Colab for demo purposes. For production, run a persistent Redis instance.
- TensorRT is not fully available on standard Colab runtimes; the notebook will install `tensorrt` client packages for integration tests.

## 1) Quick environment check (Colab)
Run this cell first in Colab to confirm GPU and CUDA availability. If running locally in WSL, open a WSL terminal and run `nvidia-smi`.

In [ ]:
# Colab quick check: shows GPU and CUDA versions
!nvidia-smi || true
import sys
print('Python', sys.version)

## 2) Install Python dependencies (Colab-ready)
This cell installs the Python packages required by `native/autoencoder/requirements.txt`. On local WSL, prefer creating a virtualenv and running `pip install -r requirements.txt`.

In [ ]:
# Colab: install runtime packages
!pip install --upgrade pip setuptools wheel -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 -q || true
!pip install transformers peft accelerate bitsandbytes datasets sentencepiece -q || true
!pip install scikit-learn redis numpy onnxruntime tritonclient[http] -q || true
!pip install tensorrt -q || true

## 3) Clone your repository (or mount from Drive)
If you want to mount from Google Drive, you can replace the git clone step with Drive mount steps. For a one-off demo we clone the repo.

In [ ]:
# Clone the repo into /content/deeds-web-app (replace URL if private)
!git clone https://github.com/semaj90/mau5law.git /content/deeds-web-app || true
%cd /content/deeds-web-app/native/autoencoder
!ls -la

## 4) Start a lightweight Redis instance (Colab)
We start `redis-server` in the background for the demo so the training scripts can push snapshots and read/write feedback.

In [ ]:
# Start redis-server in background (Colab). For local, run `redis-server` in WSL or local Docker.
!apt-get update -y -qq || true
!apt-get install -y -qq redis-server || true
!redis-server --daemonize yes || true
!redis-cli ping || true

## 5) Prepare a small embeddings demo file (optional)
If you already have embeddings (PyTorch `.pt`) place them in `data/embeddings.pt`. This cell will create a tiny synthetic embeddings file for a fast demo.

In [ ]:
# Create a tiny synthetic embeddings tensor for demo purposes
python_code = '''
import torch
import os
os.makedirs('data', exist_ok=True)
emb = torch.randn(256, 768)  # 256 vectors of 768 dims for quick run
torch.save(emb, 'data/embeddings.pt')
print('Wrote data/embeddings.pt (shape)', emb.shape)
'''
open('make_demo_embeddings.py', 'w').write(python_code)
!python make_demo_embeddings.py

## 6) Train the autoencoder (+ optional QLoRA adapter)
This runs the `train_autoencoder.py` script in the repo. For a fast Colab demo we use reduced epochs and batch sizes. Locally, increase `--epochs` and `--batch-size` and ensure CUDA + cuDNN + TensorRT are installed.

In [ ]:
# Train autoencoder (demo): writes artifacts/ with som_autoencoder.pt and foaf_latents.pt
!python train_autoencoder.py data/embeddings.pt artifacts --epochs 2 --batch-size 64 --latent-dim 128 --train-adapter || true
!ls -la artifacts || true

In [ ]:
# Plot training progress if logs exist
import json, os
import matplotlib.pyplot as plt
loss_path = 'artifacts/loss_log.json'
if os.path.exists(loss_path):
    with open(loss_path, 'r') as f:
        data = json.load(f)
    epochs = list(range(1, len(data.get('train', [])) + 1))
    plt.plot(epochs, data.get('train', []), label='train')
    if data.get('val'):
        plt.plot(epochs, data.get('val', []), label='val')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print('No artifacts/loss_log.json found. Rerun training with logging enabled.')


## 7) Run cluster trainer to generate centroids and push to Redis
This creates a KMeans snapshot and stores centroids in Redis keys the orchestrator expects.

In [ ]:
# Run cluster trainer to produce centroids and push to Redis
!python cluster_trainer.py artifacts/foaf_latents.pt artifacts --k 16 --redis-url redis://localhost:6379/0 || true
# Preview Redis keys and centroids (if present)
python - <<'PY'
import redis, json
r = redis.Redis()
print('Redis keys sample:', r.keys('foaf:*')[:20])
cent = r.get('foaf:kmeans:centroids')
if cent:
    try:
        centroids = json.loads(cent)
        print('Centroids count:', len(centroids))
    except Exception as e:
        print('Could not parse centroids:', e)
else:
    print('No centroids found in Redis under foaf:kmeans:centroids')
PY

## 8) Start the GRPO feedback worker (demo)
This will run the `scripts/rl_train_worker.py` script which listens for `rl.feedback` messages in Redis and simulates adapter updates.

In [ ]:
# Start a demo RL worker in the background (prints adapter updates)
!python ../../scripts/rl_train_worker.py --redis-url redis://localhost:6379/0 --adapter-path artifacts/adapter_legal_qlora &>/content/rl_worker.log &
!sleep 1
!tail -n 20 /content/rl_worker.log || true

## 9) Optional: Build the native libtorch binary (WSL/local)
Building the native C++ binary (`som_autoencoder`) requires libtorch. On local WSL/Ubuntu run the CMake steps below. Colab GPU runtimes typically do not provide system-wide libtorch headers for compiling C++ extensions.

In [ ]:
# Local WSL build steps (example)
echo 'Run these in WSL / Ubuntu:'
echo 'mkdir -p native/autoencoder/build && cd native/autoencoder/build'
echo 'cmake -DCMAKE_PREFIX_PATH=/path/to/libtorch ..'
echo 'cmake --build . --config Release'

## 10) Local (WSL) quick commands
If you prefer to run locally on your RTX 3060 Ti under WSL2/Ubuntu, follow these steps:

1. Create a Python venv and activate it

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r native/autoencoder/requirements.txt
# Verify CUDA / Torch
python -c 

```

2. Train (example)

```bash
python native/autoencoder/train_autoencoder.py data/embeddings.pt native/autoencoder/artifacts --train-adapter
python native/autoencoder/cluster_trainer.py native/autoencoder/artifacts/foaf_latents.pt native/autoencoder/artifacts --redis-url redis://localhost:6379/0
```

## 11) Next steps & Troubleshooting
- If the Triton/TensorRT steps fail on Colab, run inference via PyTorch (ONNX) or the native libtorch binary locally.
- After artifacts are produced, set `TRITON_URL` in your SvelteKit `.env` and the orchestrator will attempt Triton before falling back to the native binary.
- Run `npx tsc -p sveltekit-frontend/tsconfig.orchestrator.json --noEmit --skipLibCheck` to catch TypeScript integration issues after you add the artifacts.

If you'd like, I can now: generate the same notebook with a richer UI (progress bars, notebook cells that stream training logs to the output), or run a local test of the first few steps. Which would you prefer?